# ETL Step

In [ ]:
import pandas as pd 
import numpy as np
from pathlib import Path
from IPython.display import display

json_files = list(Path('data').glob('*.json'))

df = pd.DataFrame()


for file in json_files:
    try:
        tempdf = pd.read_json(file)
        df = pd.concat([df, tempdf], ignore_index=True)
        print(f"Appended {file.name} to dataframe.")
    except Exception as e:
        print(f"Error processing {file}: {e}")

df = df.drop(columns=[
    'ip_addr','master_metadata_track_name', 'audiobook_title',
    'spotify_track_uri', 'episode_name','episode_show_name',
    'spotify_episode_uri','offline_timestamp', 'incognito_mode',
    'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title',
    
])

df.rename(columns={
    'ts': 'Timestamp',
    'platform': 'Platform',
    'ms_played': 'Time Pleyed (ms)',
    'conn_country': 'Country',
    'master_metadata_album_artist_name': 'Artist Name',
    'master_metadata_album_album_name': 'Album Name',
    'reason_start': 'Reason Started',
    'reason_end': 'Reason Ended',
    'shuffle': 'Shuffle',
    'skipped': 'Skipped',
    'offline': 'Offline'
}, inplace=True)

df['Platform Main'] = df['Platform'].str.split(' ').str[0]
df['Platform Main'].replace({
    'Partner': 'Android TV',
    'web_player': 'Web Player',
    'WebPlayer': 'Web Player'
}, inplace=True)

df['Platform Main'] = df['Platform Main'].str.upper()
df['Time Pleyed (ms)'] = pd.to_timedelta(df['Time Pleyed (ms)'], unit='ms')
df['Length_min'] = df['Time Pleyed (ms)'].dt.total_seconds() / 60

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Year'] = df['Timestamp'].dt.year


Appended Streaming_History_Audio_2014-2019_0.json to dataframe.
Appended Streaming_History_Audio_2019-2020_1.json to dataframe.
Appended Streaming_History_Audio_2020-2021_2.json to dataframe.
Appended Streaming_History_Audio_2021-2022_3.json to dataframe.
Appended Streaming_History_Audio_2022-2023_4.json to dataframe.
Appended Streaming_History_Audio_2023-2024_5.json to dataframe.
Appended Streaming_History_Audio_2024-2025_6.json to dataframe.
Appended Streaming_History_Audio_2025-2026_9.json to dataframe.
Appended Streaming_History_Audio_2025_7.json to dataframe.
Appended Streaming_History_Audio_2025_8.json to dataframe.
Appended Streaming_History_Video_2020-2026.json to dataframe.


# Platform Analysis
This section will anser to question which platform was utilized the most by me

In [115]:
platform_pivot = df.pivot_table(index=['Platform Main'], values='Length_min', aggfunc='sum').sort_values(by='Length_min', ascending=False)

'''
I was quite supries with the amount of time I have spent on Spotify in total
That's why there are calcualtion up to years, just to get a better perspective on the amount of time I have spent there
'''

platform_pivot['Length_min'] = platform_pivot['Length_min'].round(2)
platform_pivot['Length_hours'] = (platform_pivot['Length_min'] / 60).round(2)
platform_pivot['Length_days'] = (platform_pivot['Length_hours'] / 24).round(2)
platform_pivot['Length_months'] = (platform_pivot['Length_days'] / 30).round(2)
platform_pivot['Length_years'] = (platform_pivot['Length_months'] / 12).round(2)
platform_pivot['Pct %'] = ((platform_pivot['Length_min'] / platform_pivot['Length_min'].sum()) * 100).round(2)

display(platform_pivot)

years_pivot = df.pivot_table(index=['Year'], values='Length_min', aggfunc='sum').sort_values(by='Year', ascending=True)
years_pivot['Length_min'] = years_pivot['Length_min'].round(2)
years_pivot['Pct %'] = ((years_pivot['Length_min'] / years_pivot['Length_min'].sum()) * 100).round(2)
years_pivot

,Length_min,Length_hours,Length_days,Length_months,Length_years,Pct %
Platform Main,,,,,,
ANDROID,213272.17,3554.54,148.11,4.94,0.41,48.41
WINDOWS,125219.27,2086.99,86.96,2.90,0.24,28.42
IOS,80889.34,1348.16,56.17,1.87,0.16,18.36
WEB PLAYER,18144.44,302.41,12.60,0.42,0.03,4.12
PLAYSTATION,1578.20,26.30,1.10,0.04,0.00,0.36
CAST,1114.75,18.58,0.77,0.03,0.00,0.25
UNKNOWN,216.40,3.61,0.15,0.00,0.00,0.05
ANDROID TV,162.54,2.71,0.11,0.00,0.00,0.04


,Length_min,Pct %
Year,,
2014,4287.43,0.97
2019,39495.21,8.96
2020,54685.14,12.41
2021,71340.20,16.19
2022,60409.64,13.71
2023,39844.42,9.04
2024,55201.41,12.53
2025,94265.88,21.40
2026,21067.78,4.78
